In [ ]:
"""
Variable-Density Spiral (2D, non-slice-selective) sequence in PyPulseq
=======================================================================

Terminology note: "spiral" and "EPI" are two different k-space trajectories.
There is no literal "spiral EPI" - what's normally meant is a fast single-
or multi-shot SPIRAL readout, which plays the same role for non-Cartesian
imaging that EPI plays for Cartesian imaging (read out a lot of k-space
per excitation). That's what this script builds.

Method
------
1. Design the SHAPE of the spiral in k-space by choosing, at every radius r,
   a local field-of-view FOV(r). Variable density = FOV(r) shrinks towards
   the edge of k-space, i.e. the outer part of k-space is undersampled
   relative to Nyquist while the centre remains fully sampled.
2. Integrate the local sampling-density condition
        dtheta/dr = 2*pi*FOV(r) / n_interleaves
   to get theta(r), hence the k-space path k(r) = r * exp(i*theta(r)).
3. Re-parametrise that path in TIME using a forward/backward
   (trapezoidal) speed-limiting pass, respecting the scanner's maximum
   gradient amplitude and slew rate (time-optimal path following).
4. Resample onto the gradient raster, differentiate to get gx(t)/gy(t).
5. Rotate the same base waveform by 2*pi/n_interleaves for each shot.

Set n_interleaves = 1 for single-shot, > 1 for multi-shot.
"""

import numpy as np
import matplotlib.pyplot as plt
import pypulseq as pp

# ---------------------------------------------------------------------
# 1. Acquisition parameters
# ---------------------------------------------------------------------
fov_x, fov_y = 256e-3, 256e-3      # [m]
n_x, n_y = 128, 128                 # matrix size

assert fov_x == fov_y and n_x == n_y, "this script assumes square FOV/matrix"
fov = fov_x
kmax = n_x / (2 * fov)              # [1/m] Nyquist edge of k-space

n_interleaves = 8       # 1 = single-shot spiral, >1 = multi-shot (stack of rotated arms)
us_edge = 3.0           # undersampling factor at the edge of k-space (>=1)
                        # us_edge = 1  -> uniform (constant) density spiral
alpha_taper = 1.0       # shape of the FOV(r) taper (1 = linear taper)

TR = 250e-3
flip_angle = 15 * np.pi / 180

# ---------------------------------------------------------------------
# 2. System limits
# ---------------------------------------------------------------------
system = pp.Opts(
    max_grad=32, grad_unit='mT/m',
    max_slew=130, slew_unit='T/m/s',
    rf_ringdown_time=20e-6,
    rf_dead_time=100e-6,
    adc_dead_time=10e-6,
    grad_raster_time=10e-6,
)

grad_raster = system.grad_raster_time      # [s]
Gmax = 0.95 * system.max_grad              # [Hz/m]  10% safety margin
Smax = 0.90 * system.max_slew              # [Hz/m/s]

# ---------------------------------------------------------------------
# 3. Design the spiral SHAPE k(r): integrate dtheta/dr
# ---------------------------------------------------------------------
def local_fov(r_norm):
    """Variable-density FOV(r); r_norm = r/kmax in [0, 1]."""
    return fov * (1 - (1 - 1 / us_edge) * r_norm ** alpha_taper)

n_shape = 20000
r = np.linspace(1e-6, kmax, n_shape)       # avoid the r=0 singularity
dtheta_dr = 2 * np.pi * local_fov(r / kmax) / n_interleaves
theta = np.concatenate(([0.0],
                        np.cumsum(0.5 * (dtheta_dr[1:] + dtheta_dr[:-1]) * np.diff(r))))
k_shape = r * np.exp(1j * theta)           # base interleaf path, [1/m]
k_shape = np.concatenate(([0.0 + 0j], k_shape))   # start exactly at k=0

# ---------------------------------------------------------------------
# 4. Time-optimal re-parametrisation (forward/backward speed limiting)
#
# A spiral path is curved, so the gradient vector g(t)=dk/dt changes
# direction as well as magnitude. The total slew vector has a tangential
# part (dv/dt, speed changing) AND a centripetal/normal part (v^2*kappa,
# direction changing, kappa = local path curvature). Both have to fit
# inside Smax together, so the Smax budget is split between them
# (1/sqrt(2) each -> combined magnitude stays <= Smax).
# ---------------------------------------------------------------------
ds = np.abs(np.diff(k_shape))              # path length per step, [1/m]
N = len(k_shape)

x, y = k_shape.real, k_shape.imag
dx, dy = np.gradient(x), np.gradient(y)
d2x, d2y = np.gradient(dx), np.gradient(dy)
denom = (dx ** 2 + dy ** 2) ** 1.5
kappa = np.abs(dx * d2y - dy * d2x) / np.maximum(denom, 1e-30)   # local curvature, [m]

a_split = Smax / np.sqrt(2)                # slew budget split tangential/normal
v_curve_max = np.sqrt(a_split / np.maximum(kappa, 1e-9))
v_cap = np.minimum(Gmax, v_curve_max)      # per-point speed cap, [Hz/m]

v_f = np.zeros(N)
v_b = np.zeros(N)
for i in range(1, N):
    v_f[i] = min(v_cap[i], np.sqrt(v_f[i - 1] ** 2 + 2 * a_split * ds[i - 1]))
for i in range(N - 2, -1, -1):
    v_b[i] = min(v_cap[i], np.sqrt(v_b[i + 1] ** 2 + 2 * a_split * ds[i]))
v = np.minimum(v_f, v_b)
v = np.maximum(v, 1e-6)                    # avoid divide-by-zero

dt = ds / (0.5 * (v[1:] + v[:-1]))
t_path = np.concatenate(([0.0], np.cumsum(dt)))

# resample onto the gradient raster
t_grid = np.arange(0, t_path[-1], grad_raster)
kx_grid = np.interp(t_grid, t_path, k_shape.real)
ky_grid = np.interp(t_grid, t_path, k_shape.imag)

# ---------------------------------------------------------------------
# 5. Differentiate -> base gradient waveform (Hz/m)
# ---------------------------------------------------------------------
gx = np.diff(kx_grid) / grad_raster
gy = np.diff(ky_grid) / grad_raster
gx = np.concatenate(([0.0], gx))
gy = np.concatenate(([0.0], gy))

# ramp both channels back down to zero at the end of the readout
ramp_pts = int(np.ceil(max(np.max(np.abs(gx)), np.max(np.abs(gy))) / Smax / grad_raster)) + 1
ramp = np.linspace(1, 0, ramp_pts)
gx = np.concatenate((gx, gx[-1] * ramp))
gy = np.concatenate((gy, gy[-1] * ramp))

n_adc = len(gx)
print(f"Spiral readout: {n_adc} samples/interleaf, "
      f"{n_adc * grad_raster * 1e3:.2f} ms/interleaf, "
      f"{n_interleaves} interleave(s), total scan {n_interleaves * TR * 1e3:.1f} ms")
print(f"max|Gx|={np.max(np.abs(gx)):.0f} Hz/m, max|Gy|={np.max(np.abs(gy)):.0f} Hz/m, "
      f"limit={system.max_grad:.0f} Hz/m")

assert np.max(np.abs(gx)) <= system.max_grad + 1, "Gx exceeds max_grad"
assert np.max(np.abs(gy)) <= system.max_grad + 1, "Gy exceeds max_grad"

# ---------------------------------------------------------------------
# 6. Build the sequence
# ---------------------------------------------------------------------
seq = pp.Sequence(system=system)

rf = pp.make_block_pulse(flip_angle=flip_angle, duration=0.5e-3, system=system)
adc = pp.make_adc(num_samples=n_adc, dwell=grad_raster, delay=system.adc_dead_time, system=system)

for shot in range(n_interleaves):
    phi = 2 * np.pi * shot / n_interleaves
    gx_i = gx * np.cos(phi) - gy * np.sin(phi)
    gy_i = gx * np.sin(phi) + gy * np.cos(phi)

    gx_shot = pp.make_arbitrary_grad(channel='x', waveform=gx_i, system=system)
    gy_shot = pp.make_arbitrary_grad(channel='y', waveform=gy_i, system=system)

    seq.add_block(rf)
    seq.add_block(pp.make_delay(system.rf_dead_time))
    seq.add_block(gx_shot, gy_shot, adc)

    t_used = pp.calc_duration(rf) + system.rf_dead_time + pp.calc_duration(gx_shot, gy_shot, adc)
    if TR > t_used:
        seq.add_block(pp.make_delay(TR - t_used))

ok, err = seq.check_timing()
print("Timing check:", "OK" if ok else err)

seq.set_definition('FOV', [fov_x, fov_y, 1])
seq.set_definition('Name', 'vd_spiral_2d')
seq.write('variable_density_spiral_2d.seq')

# ---------------------------------------------------------------------
# 7. Quick plots
# ---------------------------------------------------------------------
gamma = 42.577e6  # Hz/T
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].plot(kx_grid, ky_grid, lw=0.7)
ax[0].set_aspect('equal')
ax[0].set_title('k-space, interleaf 0')
ax[0].set_xlabel('kx [1/m]')
ax[0].set_ylabel('ky [1/m]')

t_ms = np.arange(len(gx)) * grad_raster * 1e3
ax[1].plot(t_ms, gx * 1e3 / gamma, label='Gx [mT/m]')
ax[1].plot(t_ms, gy * 1e3 / gamma, label='Gy [mT/m]')
ax[1].set_xlabel('time [ms]')
ax[1].legend()
ax[1].set_title('Gradients (interleaf 0)')
plt.tight_layout()
plt.savefig('spiral_check.png', dpi=150)
print("Done.")